In [1]:
import os
from dotenv import load_dotenv

# 1. Load the hidden environment variables
load_dotenv()  

# 2. Fetch the key (make sure there is no space between GROQ and _API_KEY)
api_key = os.getenv("GROQ_API_KEY")

# 3. Quick test to confirm it loaded successfully
if api_key:
    print("Success: API Key loaded!")
    print(f"Key starts with: {api_key[:8]}...") # Shows 'gsk_xxxx' without revealing the whole key
else:
    print("Error: Could not find GROQ_API_KEY. Check your .env file spelling!")

Success: API Key loaded!
Key starts with: gsk_MRMG...


In [2]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [3]:
import os 
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
llm_model = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature= 0.6,
    max_retries=2
)

In [5]:
prompt_template = PromptTemplate(
    input_variables= ['cuisine'],
    template= "I want to open a restaurant for {cuisine} food. Suggest a fancy name for this, only one name, no explination."
)
prompt_template.template.format(cuisine="mexican")

'I want to open a restaurant for mexican food. Suggest a fancy name for this, only one name, no explination.'

In [6]:
llm_restaurant_name_chain = prompt_template | llm_model | StrOutputParser()

llm_response_1 = llm_restaurant_name_chain.invoke({"cuisine" : "mexican"})
print(llm_response_1)

"El Fuego de Oro"


In [7]:
# Define LLM
llm_model = llm_model

# Step 1: Restaurant name chain
prompt_template = PromptTemplate(
    input_variables= ['cuisine'],
    template= "I want to open a restaurant for {cuisine} food. Suggest a fancy name for this. Return ONLY one name, no explanation."
)

llm_restaurant_name_chain = prompt_template | llm_model | StrOutputParser() 

# Step 2: Menu chain
prompt_template = PromptTemplate(
    input_variables= ['restaurant_name'],
    template= """Suggest menu items for {restaurant_name}. Return ONLY a comma-separated list, no explanations, no translations."""
)

llm_menu_chain = prompt_template | llm_model | StrOutputParser()
 

In [8]:
# overall_chain = llm_restaurant_name_chain | llm_menu_chain | StrOutputParser()

# llm_response_2 = overall_chain.invoke({"cuisine":"german"})
# print(llm_response_2)

In [9]:
# # Full sequential pipeline (replaces SimpleSequentialChain)
# full_chain = llm_restaurant_name_chain | (lambda name: {"menu_card": name}) | llm_menu_chain

# result = full_chain.invoke({"cuisine": "Italian"})
# print(result)

In [13]:
from langchain_core.runnables import RunnableLambda

def run_pipeline(input_dict):
    restaurant_name = llm_restaurant_name_chain.invoke({"cuisine": input_dict["cuisine"]})
    menu_card = llm_menu_chain.invoke({"restaurant_name": restaurant_name})
    return {"restaurant_name": restaurant_name, "menu_card": menu_card}

full_chain = RunnableLambda(run_pipeline)

result = full_chain.invoke({"cuisine": "Spanish"})
print(result)
# {'restaurant_name': 'Bella Fortuna', 'menu_card': 'Bruschetta, Margherita Pizza, ...'}

{'restaurant_name': 'El Toro Dorado', 'menu_card': 'Tacos al pastor, Carne asada burritos, Chiles rellenos, Sopaipillas, Empanadas, Chiles en nogada, Enchiladas rojas, Quesadillas de pollo, Tostadas de ceviche, Flan, Churros con cajeta.'}
